# 01 · Preparación de Datos para el Modelo
### Limpieza · Imputación · Feature Engineering · Encoding · Split
---
Este notebook corresponde a la fase de preparación de datos antes de entrenar el modelo.

Este cuadernillo documenta el pipeline de preparación de datos que usa `src/preprocessing.py`.


Deteccion de la raíz del proyecto

In [1]:
import sys, os
# Detectar la raíz del proyecto automáticamente
# Sube desde models/ hasta encontrar la carpeta que tiene src/
_here = os.getcwd()  # carpeta donde está el notebook
ROOT = _here
for _ in range(5):  # sube hasta 5 niveles
    if os.path.isdir(os.path.join(ROOT, 'src')):
        break
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)
print(f'ROOT detectado: {ROOT}')

ROOT detectado: c:\Users\PC\Downloads\Proyecto_ML


Importación de librerías

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.data_loader import DataLoader
from src.preprocessing import ProcesadorDatos

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 110})

# Cargar datos
loader = DataLoader('../data/indian_roads_dataset.csv')
df_raw = loader.cargar_datos()
procesador = ProcesadorDatos()


## 1.1 Imputación de nulos

In [3]:
df = procesador.imputar_nulos(df_raw)

In [4]:
display(pd.DataFrame({'Variable':['festival'],'Nulos antes':[int(df_raw['festival'].isnull().sum())],'Nulos después':[0],'Acción':['Imputado como none']}).style.set_caption('Imputación de nulos').hide(axis='index'))

Variable,Nulos antes,Nulos después,Acción
festival,19885,0,Imputado como none


Se identificó que la variable festival tenía registros nulos, y esos valores fueron imputados como none. Al tratarse de una variable categórica, dejarla vacía generaría confusión para el modelo. Asi al asignarle una categoría explícita "None" permite conservar los registros y evitar pérdida en la información.

## 1.2 Feature Engineering (Ingeniería de características)

| Variable | Lógica | Descripción |
|----------|--------|-------------|
| `night_drive` | hora ≥ 20 o ≤ 5 | Conducción nocturna |
| `rush_hour` | 7–9h o 17–19h | Hora pico |
| `bad_weather` | fog o rain | Clima adverso |
| `low_vis` | visibility == low | Visibilidad reducida |
| `high_traffic` | traffic_density == high | Tráfico denso |
| `high_risk_cause` | drunk driving / overspeeding | Causa de alto riesgo |
| `is_highway` | road_type == highway | Autopista |
| `hora_sin / hora_cos` | ciclo 24h | Codificación cíclica de la hora |
| `noche_lluvia` | night × bad_weather | Interacción noche + lluvia |
| `vis_trafico` | low_vis × high_traffic | Interacción visibilidad + tráfico |
| `pico_trafico` | rush_hour × high_traffic | Interacción hora pico + tráfico |
| `risk_score` | del CSV (automático) | Indicador de riesgo vial |


Se hizo un estudio para contruir variables de factores de riesgo en accidentes de transito.

Y evidenciamos que variables como por ejemplo `weather: sunny, rainy, foogy, etc` era una variable categorica compleja, para no dejar muchas categorias creamos la variable `bad_weather: sí o no`. 

Asi sucesivamente con las otras variables, ya que sabemos que el riesgo real no depende de una sola variable si no de varias combinaciones.

Ejemplo:

Noche sola → riesgo medio / Lluvia sola → riesgo medio / Noche + lluvia → riesgo alto 

Lo que permite al modelo capturar situaciones más dificiles y evitamos el overfitting.

In [5]:
df = procesador.crear_features(df)

nuevas = ['night_drive','rush_hour','bad_weather','low_vis',
          'high_traffic','high_risk_cause','is_highway']
display(pd.DataFrame([{'Variable':v,'N=1':int(df[v].sum()),'%':round(df[v].mean()*100,1)} for v in nuevas]).style.set_caption('Variables engineered').hide(axis='index'))
display(df['risk_score'].describe().round(3).to_frame().T.style.set_caption('Estadísticas de risk_score'))


Variable,N=1,%
night_drive,8389,41.900000
rush_hour,4945,24.700000
bad_weather,13310,66.600000
low_vis,9987,49.900000
high_traffic,7034,35.200000
high_risk_cause,8003,40.000000
is_highway,6616,33.100000


,count,mean,std,min,25%,50%,75%,max
risk_score,20000.000000,0.438000,0.218000,0.100000,0.250000,0.450000,0.600000,1.000000


En esta etapa se aplicó feature engineering para construir variables más informativas a partir de los datos originales. La idea fue crear variables que influyan en la severidad o probabilidad de un accidente, como conducción nocturna, clima adverso, tráfico alto o causas de alto riesgo. 

Este cuadro muestra la distribución de las variables creadas. Permite entender qué tan frecuentes son ciertas condiciones como conducción nocturna, mal clima o tráfico alto.

Esto es importante porque ayuda a identificar qué variables pueden tener mayor impacto en el modelo y también a detectar si hay desbalance en los datos.

Las estadísticas del risk_score muestran que la variable tiene una distribución adecuada, con valores que van desde 0.1 hasta 1.0 y una media de 0.438. Esto indica que hay suficiente variabilidad en los datos, lo cual es positivo para el modelo, ya que permite diferenciar entre niveles de riesgo. 

## 1.3 Encoding y Split

In [6]:
X, y = procesador.preparar_datos(df_raw)

display(pd.DataFrame({'Dimensión':['Registros','Features'],'Valor':[X.shape[0],X.shape[1]]}).style.set_caption('Dimensiones tras encoding').hide(axis='index'))

Dimensión,Valor
Registros,20000
Features,43


Se transforman las variables categoricas en números para que el modelo pueda entenderlas.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

display(pd.DataFrame({'Conjunto':['Train','Test'],'Registros':[f'{len(X_train):,}',f'{len(X_test):,}'],'%':['80%','20%']}).style.set_caption('Split 80/20 estratificado').hide(axis='index'))

Conjunto,Registros,%
Train,"16,000",80%
Test,"4,000",20%


Después de preparar las variables, se separó el dataset en entrenamiento y prueba con una proporción 80/20.

Esta división permite entrenar el modelo con una parte de los datos y evaluar su desempeño con otra parte que no ha visto antes.